## Testing at different $L^A_{eff}$

In [1]:
# ================================
# eLoss-only notebook (pub styling)
# ================================
import numpy as np, matplotlib.pyplot as plt, sys
from dataclasses import replace
sys.path.append("../code")

from glauber import SystemSpec, OpticalGlauber
from quenching import QuenchParams, xA0_from_L, qhat_of_x, _l2, dpt_side
from cross_sections import (SigmaPPTable, map_sigma, map_R, sigma_avg_in_bin, RpA_vs_centrality, plot_r_vs_y_in_bins, plot_r_vs_pt_in_bins, average_R,
                            L_eff_minbias)
# your Particle and alpha_s providers:
from particle import Particle              # assumes .d2sigma_pp, .mT, .x1, .x2, .y_max, .xA, .xB implemented
from coupling import alpha_s_provider      # running/constant αs factory

In [2]:
# --- 0) Knobs (edit here) ---
# roots = 8160.0
roots = 5023.0
A     = 208
sigma_nn_mb = 67.6 if roots == 5023.0 else 71.0
# sigma_nn_mb8pPb = 71.0

# analysis windows (you can narrow if you want it even faster)
Y_EDGES  = np.linspace(-5.0, 5.0, 21)         # 20 bins in y
PT_EDGES = np.linspace( 0.0, 20.0, 41)        # 40 bins in pT

# qhat0 band (center value is 0.07; band shown using min..max)
Q0_LIST = (0.05, 0.07, 0.09)

# numerics for averaging (Gauss-Legendre nodes along y and pT)
Ny_bin = 20
Npt_bin = 48

# Your defined windows and labels
Y_WINDOWS = [(-4.46,-2.96), (-1.37,0.43), (2.03,3.53)]
Y_LABELS  = ["-4.46 < y < -2.96", "-1.37 < y < 0.43", "2.03 < y < 3.53"]
# Tags for unique filenames
Y_FILES   = ["Back", "Mid", "For"]

# choose αs: constant 0.5 (AP default) vs running
USE_RUNNING = False

In [3]:
# --- 1) Systems & Glauber (L_Aeff) ---
spec_pA = SystemSpec(system="pA", roots_GeV=roots, A=A, sigma_nn_mb=sigma_nn_mb)
gl      = OpticalGlauber(spec_pA, verbose=False)
LAmb    = L_eff_minbias(gl, "pA")           # [fm]
# LAmb    = gl.leff_minbias_pA()
print(f"[Glauber] L_eff^MB (pA) = {LAmb:.3f} fm")

# --- 2) Particles ---
P_charmonia = Particle(family="charmonia",  state="avg")
P_bottomonia = Particle(family="bottomonia", state="avg")

# --- 3) αs provider ---
from coupling import alpha_s_provider
alpha0 = 0.5 ## 0.326
alpha_cst = alpha_s_provider(mode="constant", alpha0=0.50)
alpha_run = alpha_s_provider(mode="running", loops=4, method="ode")
alpha_s   = alpha_run if USE_RUNNING else alpha_cst
print("[Coupling] Coupling used: ", alpha_s)

[Glauber] L_eff^MB (pA) = 10.414 fm
[Coupling] Coupling used:  <function alpha_s_provider.<locals>.<lambda> at 0x76717269d940>


In [4]:
# pp tables (cache σ_pp once per (particle,energy))
def pp_table(P, roots):
    y_tab  = np.linspace(-5.0, 5.0, 41)
    pt_tab = np.linspace( 0.0, 25.0, 101)
    return SigmaPPTable(P, roots, y_tab, pt_tab)
    
# QuenchParams template — we’ll only change qhat0 and L_A/B per run
from quenching import QuenchParams
def qpar_template(roots, LA):
    return QuenchParams(
        qhat0 = 0.07,
        lp_fm = 1.5,
        LA_fm = float(LA),
        LB_fm = float(LA),
        lambdaQCD = 0.25, # 0.25, 0.308
        roots_GeV = float(roots),
        alpha_of_mu = alpha_s,
        alpha_scale = "mT"
    )

In [5]:
# Lightweight bilinear interpolator on a rect grid
class Bilin:
    def __init__(self, x, y, Z):
        self.x, self.y, self.Z = np.asarray(x), np.asarray(y), np.asarray(Z)
    def __call__(self, xq, yq):
        xi = np.clip(np.searchsorted(self.x, xq)-1, 0, len(self.x)-2)
        yi = np.clip(np.searchsorted(self.y, yq)-1, 0, len(self.y)-2)
        x0,x1 = self.x[xi], self.x[xi+1]
        y0,y1 = self.y[yi], self.y[yi+1]
        z00 = self.Z[xi,   yi  ]; z10 = self.Z[xi+1, yi  ]
        z01 = self.Z[xi,   yi+1]; z11 = self.Z[xi+1, yi+1]
        tx = (xq - x0)/max(x1-x0, 1e-15)
        ty = (yq - y0)/max(y1-y0, 1e-15)
        return (1-tx)*(1-ty)*z00 + tx*(1-ty)*z10 + (1-tx)*ty*z01 + tx*ty*z11

In [6]:
# Define centrality bins (fractions)
cent_edges = (0.0, 0.2, 0.4, 0.6, 0.8, 1.0)   # 0-20-40-60-80-100%

# Glauber (pPb @ LHC), then get L_eff
spec8 = SystemSpec(system="pA", roots_GeV=8160.0, A=208, sigma_nn_mb=71.0)
spec5 = SystemSpec(system="pA", roots_GeV=5023.0, A=208, sigma_nn_mb=67.6)
gl8 = OpticalGlauber(spec8, verbose=True)
gl5 = OpticalGlauber(spec5, verbose=True)
# Centrality-binned L_eff using A&P binomial method (Eq. B.9)
bins = [(0,20),(20,40),(40,60),(60,80),(80,100)]
LA_by_bin_8pA = gl8.leff_bins_pA(bins, rho0_fm3=0.17, Lp_fm=1.5, method="binomial")
LA_by_bin_5pA = gl5.leff_bins_pA(bins, rho0_fm3=0.17, Lp_fm=1.5, method="binomial")

# Optical L_Eff
LA_by_bin_opt_8pA = gl8.leff_bins_pA(bins, method="optical")
LA_by_bin_opt_5pA = gl5.leff_bins_pA(bins, method="optical")
# print(LA_by_bin_opt_8pA)

L_eff_results = {
    '8.16TeV': {
        'Binomial': gl8.leff_bins_pA(bins, rho0_fm3=0.17, Lp_fm=1.5, method="binomial"),
        'Optical':  gl8.leff_bins_pA(bins, method="optical")
    },
    '5.02TeV': {
        'Binomial': gl5.leff_bins_pA(bins, rho0_fm3=0.17, Lp_fm=1.5, method="binomial"),
        'Optical':  gl5.leff_bins_pA(bins, method="optical")
    }
}

print("-" * 80)
print(f"{'Centrality':<12} | {'Binom (8 TeV)':>13} | {'Binom (5 TeV)':>13} | {'Opt (8 TeV)':>12} | {'Opt (5 TeV)':>12}")
print("-" * 80)
bin_keys = [f"{b[0]}-{b[1]}%" for b in bins]
# Loop using the bin keys to ensure order
for key in bin_keys:
    b8 = L_eff_results['8.16TeV']['Binomial'].get(key, 'N/A')
    b5 = L_eff_results['5.02TeV']['Binomial'].get(key, 'N/A')
    o8 = L_eff_results['8.16TeV']['Optical'].get(key, 'N/A')
    o5 = L_eff_results['5.02TeV']['Optical'].get(key, 'N/A')

    print(f"| {key:<12} | {b8:>13.3f} | {b5:>13.3f} | {o8:>12.3f} | {o5:>12.3f} |")
print("-" * 80)

[Glauber] TA(r) LUT: A=208 d=0.549 r≤50 fm, dr=0.02, z≤50 fm
[Glauber] ∫T_A d^2x ≈ 208.483 (target A=208)
[Glauber] Tabulating T_AA(b), T_pA(b)…
[Glauber] σ_tot^AA ≈ 7757.76 mb, σ_tot^pA ≈ 1925.90 mb
[Glauber] TA(r) LUT: A=208 d=0.549 r≤50 fm, dr=0.02, z≤50 fm
[Glauber] ∫T_A d^2x ≈ 208.483 (target A=208)
[Glauber] Tabulating T_AA(b), T_pA(b)…
[Glauber] σ_tot^AA ≈ 7724.42 mb, σ_tot^pA ≈ 1911.63 mb
--------------------------------------------------------------------------------
Centrality   | Binom (8 TeV) | Binom (5 TeV) |  Opt (8 TeV) |  Opt (5 TeV)
--------------------------------------------------------------------------------
| 0-20%        |        14.109 |        13.918 |       13.305 |       13.305 |
| 20-40%       |         9.455 |         9.380 |       10.775 |       10.811 |
| 40-60%       |         6.217 |         6.459 |        7.653 |        7.748 |
| 60-80%       |         3.222 |         3.315 |        4.648 |        4.706 |
| 80-100%      |         1.500 |         1.500 

In [7]:
# --- Example: Accessing a specific value by key ---
print("--- Example Access ---")
key = '0-20%'
print(f"L_eff (Optical, 8.16TeV, {key}): {L_eff_results['8.16TeV']['Optical'][key]:.3f}\n")

--- Example Access ---
L_eff (Optical, 8.16TeV, 0-20%): 13.305



In [8]:
# --- 5) σ_pp cache for speed ---
from cross_sections import SigmaPPTable
_y_tab  = np.linspace(-5.0, 5.0, 41)       # coarse but safe
_pt_tab = np.linspace( 0.0, 25.0, 101)
PP_TAB_CHI = SigmaPPTable(P_charmonia, roots, _y_tab, _pt_tab)
PP_TAB_BOT = SigmaPPTable(P_bottomonia, roots, _y_tab, _pt_tab)

# --- 6) Helpers: per-bin RpA with q0 band (single L_A_eff) ---
from cross_sections import average_R

def _bin_avg_R_band(P, table, y_rng, pt_rng, q0_list=Q0_LIST, Ny=Ny_bin, Npt=Npt_bin):
    vals=[]
    for q0 in q0_list:
        qp = replace(QP_TEMPLATE, qhat0=float(q0))
        vals.append(average_R(P, roots, qp, y_rng, pt_rng, Ny=Ny, Npt=Npt,
                              kind="pA", table=table))
    arr = np.asarray(vals, float)
    return float(arr.mean()), float(arr.min()), float(arr.max())

def band_vs_y(P, table, y_edges, pt_range):
    mids, mean, lo, hi = [], [], [], []
    for yl, yr in zip(y_edges[:-1], y_edges[1:]):
        m, a, b = _bin_avg_R_band(P, table, (yl,yr), pt_range)
        mids.append(0.5*(yl+yr)); mean.append(m); lo.append(a); hi.append(b)
    return np.array(mids), np.array(mean), np.array(lo), np.array(hi)

def band_vs_pt(P, table, y_range, pt_edges):
    mids, mean, lo, hi = [], [], [], []
    for pl, pr in zip(pt_edges[:-1], pt_edges[1:]):
        m, a, b = _bin_avg_R_band(P, table, y_range, (pl,pr))
        mids.append(0.5*(pl+pr)); mean.append(m); lo.append(a); hi.append(b)
    return np.array(mids), np.array(mean), np.array(lo), np.array(hi)

In [9]:
# # --- 7) Compute bands (in chosen L_A_eff) ---
# print("[run] computing RpA(y) band in a L_A_Eff…")
# y_mid_chi, y_mean_chi, y_lo_chi, y_hi_chi = band_vs_y(P_charmonia, PP_TAB_CHI, Y_EDGES, (0.0, 20.0))
# y_mid_bot, y_mean_bot, y_lo_bot, y_hi_bot = band_vs_y(P_bottomonia, PP_TAB_BOT, Y_EDGES, (0.0, 20.0))

# print("[run] computing RpA(pT) band in a L_A_Eff …")
# pt_mid_chi, pt_mean_chi, pt_lo_chi, pt_hi_chi = band_vs_pt(P_charmonia, PP_TAB_CHI, (-5.0, 5.0), PT_EDGES)
# pt_mid_bot, pt_mean_bot, pt_lo_bot, pt_hi_bot = band_vs_pt(P_bottomonia, PP_TAB_BOT, (-5.0, 5.0), PT_EDGES)

In [10]:
# # --- 8) Plot: ONE figure, two panels; both families overlaid with q0 bands ---
# fig, (axY, axP) = plt.subplots(1, 2, figsize=(11.5, 4.6), constrained_layout=True)

# # Left: RpA vs y (pT-integrated)
# axY.fill_between(y_mid_chi, y_lo_chi, y_hi_chi, alpha=0.25, step="mid", label="Charmonia (q̂₀ band)")
# axY.step(y_mid_chi, y_mean_chi, where="mid", lw=2)
# axY.fill_between(y_mid_bot, y_lo_bot, y_hi_bot, alpha=0.25, step="mid", label="Bottomonia (q̂₀ band)")
# axY.step(y_mid_bot, y_mean_bot, where="mid", lw=2)
# axY.axhline(1.0, ls="--", lw=1, alpha=0.6)
# axY.set(xlabel=r"$y$", ylabel=r"$R_{pA}$")
# axY.text(0,0.6,r"$0<p_T<20$ GeV", style='italic', bbox={'facecolor': 'red', 'alpha': 0.3, 'pad': 10})
# axY.set_ylim(0.4, max(1.35, 1.05*max(y_hi_chi.max(), y_hi_bot.max())))
# axY.legend(frameon=False)

# # Right: RpA vs pT (y-integrated)
# axP.fill_between(pt_mid_chi, pt_lo_chi, pt_hi_chi, alpha=0.25, step="mid", label="Charmonia (q̂₀ band)")
# axP.step(pt_mid_chi, pt_mean_chi, where="mid", lw=2)
# axP.fill_between(pt_mid_bot, pt_lo_bot, pt_hi_bot, alpha=0.25, step="mid", label="Bottomonia (q̂₀ band)")
# axP.step(pt_mid_bot, pt_mean_bot, where="mid", lw=2)
# axP.axhline(1.0, ls="--", lw=1, alpha=0.6)
# axP.set(xlabel=r"$p_T$ [GeV]", ylabel=r"$R_{pA}$")
# axP.text(10,0.8,r"$-5<y<5$", style='italic', bbox={'facecolor': 'red', 'alpha': 0.3, 'pad': 10})
# axP.set_xlim(PT_EDGES[0], PT_EDGES[-1])
# axP.set_ylim(0.4, max(1.35, 1.05*max(pt_hi_chi.max(), pt_hi_bot.max())))
# axP.legend(frameon=False)

# fig.suptitle(
#     rf"Min-bias p–Pb @ √sNN={roots/1000:.2f} TeV  |  $L_{{\rm A}}^{{\rm MB}}$={LAmb:.2f} fm  |  "
#     + (r"$\alpha_s$ running" if USE_RUNNING else r"$\alpha_s=0.5$ const")
#     + r"  |  $q̂_0\in[0.05,0.09]$ GeV$^2$/fm",
#     y=1.05, fontsize=11
# )
# plt.show()

In [11]:
# Convenience: compute raw σ_pp, σ_pA, and RpA maps + interpolants for one (P,roots,L)
def build_maps_and_interps(P, roots, LA, pp_tab):
    qp   = qpar_template(roots, LA)
    ymid = Y_CENTERS; ptmid = PT_CENTERS
    # raw surfaces on bin centers (fast & good for plotting/interp)
    Zpp  = map_sigma(P, roots, qp, ymid, ptmid, kind="pp", table=pp_tab)
    ZpA  = map_sigma(P, roots, qp, ymid, ptmid, kind="pA", table=pp_tab)
    ZR   = map_R    (P, roots, qp, ymid, ptmid, kind="pA", table=pp_tab)

    # bilinear interpolants usable anywhere in the domain
    pp_interp  = Bilin(ymid, ptmid, Zpp)
    pA_interp  = Bilin(ymid, ptmid, ZpA)
    RpA_interp = Bilin(ymid, ptmid, ZR)
    return dict(Zpp=Zpp, ZpA=ZpA, ZR=ZR,
                pp=pp_interp, pA=pA_interp, RpA=RpA_interp, qp=qp)

In [12]:
# Bin-averaged curves (with σ_pA(y=0,pt)*pt weight)
def rpa_vs_y_binned(P, roots, qp, pp_tab, y_edges=Y_EDGES, pt_range=(0.0,20.0), Ny=Ny_bin, Npt=Npt_bin):
    vals=[]
    for yl, yr in zip(y_edges[:-1], y_edges[1:]):
        vals.append( average_R(P, roots, qp, (yl,yr), pt_range, Ny=Ny, Npt=Npt,
                               kind="pA", table=pp_tab) )
    return np.asarray(vals)

def rpa_vs_pt_binned(P, roots, qp, pp_tab, pt_edges=PT_EDGES, y_range=(-5.0,5.0), Ny=Ny_bin, Npt=Npt_bin):
    vals=[]
    for pl, pr in zip(pt_edges[:-1], pt_edges[1:]):
        vals.append( average_R(P, roots, qp, y_range, (pl,pr), Ny=Ny, Npt=Npt,
                               kind="pA", table=pp_tab) )
    return np.asarray(vals)

In [13]:
# -- knobs --
ENERGIES = (5023.0, 8160.0)         # GeV
# L_LIST   = (2.0, 5.0, 10.41, 12.0)        
L_LIST = np.sort(list(LA_by_bin_opt_8pA.values()))
Y_EDGES  = np.linspace(-5.0,  5.0, 41)   # 40 bins
PT_EDGES = np.linspace( 0.0, 20.0, 81)   # 80 bins
Y_CENTERS  = 0.5*(Y_EDGES[1:]+Y_EDGES[:-1])
PT_CENTERS = 0.5*(PT_EDGES[1:]+PT_EDGES[:-1])

In [14]:
# =========================
# Run all combinations
# =========================
results = {}  # keyed by (family, roots, L)
for roots in ENERGIES:
    tab_chi = pp_table(P_charmonia, roots)
    tab_bot = pp_table(P_bottomonia, roots)
    for L in L_LIST:
        results[("charmonia", roots, L)] = build_maps_and_interps(P_charmonia, roots, L, tab_chi)
        results[("bottomonia", roots, L)] = build_maps_and_interps(P_bottomonia, roots, L, tab_bot)

print("[ok] built (pp, pA, RpA) maps + interpolants for two energies, two families, three L_eff values.")

[ok] built (pp, pA, RpA) maps + interpolants for two energies, two families, three L_eff values.


In [15]:
# =========================
# Example plots (quicklook)
# =========================
def quicklook_surfaces(fam, roots, L, vmin=None, vmax=None):
    D = results[(fam, roots, L)]
    fig,axs = plt.subplots(1,3, figsize=(12,3.5), constrained_layout=True)
    for ax, Z, title in zip(axs, (D["Zpp"], D["ZpA"], D["ZR"]),
                            (r"$\sigma_{pp}$", r"$\sigma_{pA}$", r"$R_{pA}$")):
        im = ax.imshow(Z.T, origin="lower",
                       extent=[Y_EDGES[0], Y_EDGES[-1], PT_EDGES[0], PT_EDGES[-1]],
                       aspect="auto", vmin=vmin, vmax=vmax)
        ax.set(xlabel="y", ylabel=r"$p_T$ [GeV]", title=f"{title} @ L_eff={L} fm, √s={roots/1000:.3g} TeV")
        fig.colorbar(im, ax=ax)
    return fig, axs

In [16]:
# quicklook_surfaces("charmonia", 8160.0, 12.0); plt.show()

In [17]:
# quicklook_surfaces("bottomonia", 8160.0, 5.0); plt.show()

In [18]:
# quicklook_surfaces("charmonia", 8160.0, 2.0); plt.show()
# quicklook_surfaces("bottomonia", 8160.0, 2.0); plt.show()

In [19]:
def binned_curves_demo(fam, roots, L,
                       y_edges=Y_EDGES, pt_edges=PT_EDGES,
                       y_range=(-5,5), pt_range=(0,20)):
    P = P_charmonia if fam=="charmonia" else P_bottomonia
    TAB = pp_table(P, roots)
    qp = qpar_template(roots, L)
    r_vs_y = rpa_vs_y_binned(P, roots, qp, TAB, y_edges=y_edges, pt_range=pt_range)
    r_vs_pt = rpa_vs_pt_binned(P, roots, qp, TAB, pt_edges=pt_edges, y_range=y_range)
    fig,(ax1,ax2) = plt.subplots(1,2, figsize=(10,3.5), constrained_layout=True)
    ax1.step(y_edges[1:], r_vs_y, where="post")
    ax1.axhline(1.0, ls="--", lw=1, alpha=0.6); ax1.set(xlabel="y", ylabel=r"$\overline{R}_{pA}$")
    ax1.text(0.02, 0.93, rf"{fam}, $\sqrt{{s}}$={roots/1000:.3g} TeV, $L_{{eff}}$={L} fm, {pt_range[0]}<pT<{pt_range[1]} GeV",
             transform=ax1.transAxes, fontsize=9, va='top', ha='left')
    ax2.step(pt_edges[1:], r_vs_pt, where="post")
    ax2.axhline(1.0, ls="--", lw=1, alpha=0.6); ax2.set(xlabel=r"$p_T$ [GeV]", ylabel=r"$\overline{R}_{pA}$")
    ax2.text(0.02, 0.93, rf"{fam}, $\sqrt{{s}}$={roots/1000:.3g} TeV, $L_{{eff}}$={L} fm, {y_range[0]}<y<{y_range[1]}",
             transform=ax2.transAxes, fontsize=9, va='top', ha='left')
    return (r_vs_y, r_vs_pt), (fig,(ax1,ax2))

In [20]:
# (vals, figs) = binned_curves_demo("bottomonia", 8160.0, 5.0)
# plt.show()

In [21]:
# (vals, figs) = binned_curves_demo("charmonia", 8160.0, 5.0, y_range=Y_WINDOWS[0], pt_range=(0,20))
# (vals, figs) = binned_curves_demo("bottomonia", 8160.0, 5.0, y_range=Y_WINDOWS[0], pt_range=(0,20))
# plt.show()

In [ ]:
(vals, figs) = binned_curves_demo("charmonia", 8160.0, LAmb, y_range=Y_WINDOWS[0], pt_range=(1.5,20)) # 2.5
plt.show()

In [ ]:
# results

In [ ]:
# =========================
# Example: fast interpolation
# =========================
# After build_maps_and_interps, query R_pA(y,pT) cheaply: 8160.0, 5023.0
charm_leff5 = results[("charmonia", 5023.0, 4.64752469601636)]
charm_leff2 = results[("charmonia", 5023.0, 2.4437634046948857)] #2.4437634046948857, 2.5
charm_leff12 = results[("charmonia", 5023.0, 10.775229950676886)]
charm_leff_mb = results[("charmonia", 5023.0, LAmb)]
print("R_pA(y=2.5,pT=7,Leff=5.0):", float(charm_leff5["RpA"](2.5, 7.0)))
print("σ_pp(y=0,pT=5,Leff=5.0): ", float(charm_leff5["pp"](0.0, 5.0)))
print("σ_pA(y=-3,pT=3,Leff=5.0):", float(charm_leff5["pA"](-3.0, 3.0)))
print("R_pA(y=2.5,pT=7,Leff=2.0):", float(charm_leff2["RpA"](2.5, 7.0)))
print("σ_pp(y=0,pT=5,Leff=2.0): ", float(charm_leff2["pp"](0.0, 5.0)))
print("σ_pA(y=-3,pT=3,Leff=2.0):", float(charm_leff2["pA"](-3.0, 3.0)))
print("R_pA(y=2.5,pT=7,Leff=12.0):", float(charm_leff12["RpA"](2.5, 7.0)))
print("σ_pp(y=0,pT=5,Leff=12.0): ", float(charm_leff12["pp"](0.0, 5.0)))
print("σ_pA(y=-3,pT=3,Leff=12.0):", float(charm_leff12["pA"](-3.0, 3.0)))

KeyError: ('charmonia', 5023.0, 10.414158247314315)

In [ ]:
# pt = 5
# for y in np.linspace(-5, -2, num=50):
#     print(fr"$sigma_{{pp}}$(y={y},pT={pt}, Leff=2.0) = ", float(charm_leff2["pp"](y, pt)))
#     print(fr"$sigma_{{pA}}$(y={y},pT={pt}, Leff=2.0) = ", float(charm_leff2["pA"](y, pt)))
#     print(fr"$R_{{pA}}$(y={y},pT={pt}, Leff=2.0) = ", float(charm_leff2["RpA"](y, pt)))

In [ ]:
## Plot 1: Multiple pT values vs. y (Vectorized)
y_values = np.linspace(-5, 5, num=50)
pT_list = [1.5, 5.0, 15.0] # List of pT values to plot

plt.figure(figsize=(8, 5))

# Outer loop: iterates over each pT curve
for pT0 in pT_list:
    # FIX: Use list comprehensions [..] instead of generator expressions (...)
    sigma_pp_values = [charm_leff5["pp"](y, pT0) for y in y_values]
    sigma_pA_values5 = [charm_leff5["pA"](y, pT0) for y in y_values]
    sigma_pA_values2 = [charm_leff2["pA"](y, pT0) for y in y_values]
    sigma_pA_values12 = [charm_leff12["pA"](y, pT0) for y in y_values]
    
    # Plot the curve
    # FIX: Removed extra ')' in the label string
    plt.plot(y_values, sigma_pp_values, '-', color='black', label=rf'$\sigma_{{pp}}, p_T = {pT0}$ GeV')
    plt.plot(y_values, sigma_pA_values5, '-', label=rf'$L_{{eff}} = 5 fm, p_T = {pT0}$ GeV')
    plt.plot(y_values, sigma_pA_values2, '--', label=rf'$L_{{eff}} = 2 fm, p_T = {pT0}$ GeV')
    plt.plot(y_values, sigma_pA_values12, ':', label=rf'$L_{{eff}} = 12 fm, p_T = {pT0}$ GeV')

plt.xlabel('y')
plt.ylabel(r'$\sigma_{pA}$')
plt.legend(bbox_to_anchor=(1.01, 1), loc='upper left', borderaxespad=0.)
plt.yscale('log')
plt.show()

In [ ]:
## Plot 1: Multiple pT values vs. y (Fixed for List Division)
y_values = np.linspace(-5, 5, num=50)
pT_list = [1.5, 5.0, 15] # List of pT values to plot

plt.figure(figsize=(8, 5))

# Outer loop: iterates over each pT curve
for pT0 in pT_list:
    
    # 1. Calculate and convert the denominator (sigma_pp) to a NumPy array.
    sigma_pp_values = np.array([charm_leff5["pp"](y, pT0) for y in y_values])
    
    # 2. Calculate the numerators and convert them to NumPy arrays.
    #    Then perform the element-wise division.
    
    # Calculate R_pA for L_eff = 5 fm
    sigma_pA_values5_array = np.array([charm_leff5["pA"](y, pT0) for y in y_values])
    R_pA_values5 = sigma_pA_values5_array / sigma_pp_values
    
    # Calculate R_pA for L_eff = 2 fm
    sigma_pA_values2_array = np.array([charm_leff2["pA"](y, pT0) for y in y_values])
    R_pA_values2 = sigma_pA_values2_array / sigma_pp_values
    
    # Calculate R_pA for L_eff = 12 fm
    sigma_pA_values12_array = np.array([charm_leff12["pA"](y, pT0) for y in y_values])
    R_pA_values12 = sigma_pA_values12_array / sigma_pp_values
    
    # Plot the R_pA curves
    # Note: Using the R_pA_values variables for plotting
    # plt.plot(y_values, R_pA_values5, '-', label=rf'$L_{{eff}} = 5 fm, p_T = {pT0}$ GeV')
    # plt.plot(y_values, R_pA_values2, '--', label=rf'$L_{{eff}} = 2 fm, p_T = {pT0}$ GeV')
    plt.plot(y_values, R_pA_values12, ':', label=rf'$L_{{eff}} = 12 fm, p_T = {pT0}$ GeV')

plt.xlabel('y')
plt.ylabel(r'$R_{pA}$') 
plt.legend(bbox_to_anchor=(1.01, 1), loc='upper left', borderaxespad=0.)
plt.show()

In [ ]:
## Plot 2: Multiple y values vs. pT (Vectorized)
pT_values = np.linspace(0, 20, num=50)
y_list = [-2.0, 0.0, 2.8] # List of y values to plot

plt.figure(figsize=(8, 5))

# Outer loop: iterates over each y curve
for y_fixed in y_list:
    sigma_pA_values = [] # Reset for each new curve
    sigma_pp_values = [] # Reset for each new curve
    
    # Inner loop: builds the points for a single curve
    for pT in pT_values:
        # Call with single (y_fixed, pT) values
        sigma_pA = charm_leff5["pA"](y_fixed, pT)
        sigma_pp = charm_leff5["pp"](y_fixed, pT)
        
        sigma_pA_values.append(sigma_pA)
        sigma_pp_values.append(sigma_pp)

    # Plot the curve *after* the inner loop is done
    plt.plot(pT_values, sigma_pA_values, '-', label=f'pA at $y = {y_fixed}$')
    plt.plot(pT_values, sigma_pp_values, '--', label=f'pp at $y = {y_fixed}$')
plt.xlabel('$p_T$ [GeV]')
plt.ylabel(r'$\sigma$')
plt.legend()
plt.yscale('log')
plt.show()

In [ ]:
L_LIST

In [ ]:
## Plot : Multiple y values vs. pT 
pt_list = np.linspace(0, 20, num=50)
# y_list = [-3.7, -2.0, 0.0, 2.0, 2.8] # List of y values to plot
y_list = [-3.7, 0.0, 2.8]

plt.figure(figsize=(8, 5))

# Outer loop: iterates over each pT curve
for y in y_list:
    
    # 1. Calculate and convert the denominator (sigma_pp) to a NumPy array.
    sigma_pp_values = np.array([charm_leff5["pp"](y, pt) for pt in pt_list])
    
    # 2. Calculate the numerators and convert them to NumPy arrays.
    #    Then perform the element-wise division.
    
    # Calculate R_pA for L_eff = 5 fm
    sigma_pA_values5_array = np.array([charm_leff5["pA"](y, pt) for pt in pt_list])
    R_pA_values5 = sigma_pA_values5_array / sigma_pp_values
    
    # Calculate R_pA for L_eff = 2 fm
    sigma_pA_values2_array = np.array([charm_leff2["pA"](y, pt) for pt in pt_list])
    R_pA_values2 = sigma_pA_values2_array / sigma_pp_values
    
    # Calculate R_pA for L_eff = 12 fm
    sigma_pA_values12_array = np.array([charm_leff12["pA"](y, pt) for pt in pt_list])
    R_pA_values12 = sigma_pA_values12_array / sigma_pp_values

    # Calculate R_pA for L_eff = L_mb fm
    sigma_pA_values_mb_array = np.array([charm_leff_mb["pA"](y, pt) for pt in pt_list])
    R_pA_values_mb = sigma_pA_values_mb_array / sigma_pp_values
    
    # Plot the R_pA curves
    # Note: Using the R_pA_values variables for plotting
    # plt.plot(pt_list, R_pA_values5, '-', label=rf'$L_{{eff}} = 4.64 fm, y = {y}$')
    # plt.plot(pt_list, R_pA_values2, '--', label=rf'$L_{{eff}} = 2.44 fm, y = {y}$')
    plt.plot(pt_list, R_pA_values_mb, '-', label=rf'$L_{{eff}} =  {LAmb} fm, y = {y}$')
    # plt.plot(pt_list, R_pA_values12, ':', label=rf'$L_{{eff}} =  13.30 fm, y = {y}$')
plt.ylim(0.0,2.0)
plt.xlabel(r'$p_T$ [GeV]')
plt.ylabel(r'$R_{pA}$') 
plt.axhline(y=1, color='black', linestyle='--', linewidth=1.5, label=r'$R_{pA} = 1$')
plt.legend(bbox_to_anchor=(1.01, 1), loc='upper left', borderaxespad=0.)
plt.show()